In [25]:
# Import Necessary Libraries
import pandas as pd
from haversine import haversine

In [26]:
# Load the filtered voyage distances data
file_path = "D:\\AIS Project\\datasets\\filtered_voyage_distances.csv"
df = pd.read_csv(file_path, low_memory=False)

In [27]:
# Convert timestamp to datetime
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')

In [28]:
# Initialize columns to store time difference and distance difference
df['Time_Difference'] = pd.Timedelta(seconds=0)
df['Distance_Difference'] = 0.0

# Function to calculate time difference and distance difference
def calculate_differences(group):
    group = group.sort_values('timestamp').reset_index(drop=True)
    for i in range(1, len(group)):
        time_diff = group.at[i, 'timestamp'] - group.at[i-1, 'timestamp']
        distance_diff = haversine((group.at[i-1, 'lat'], group.at[i-1, 'lon']), 
                                  (group.at[i, 'lat'], group.at[i, 'lon']))
        group.at[i, 'Time_Difference'] = time_diff
        group.at[i, 'Distance_Difference'] = distance_diff
    return group

In [29]:
# Apply the function to each group of mmsi and Voyage_ID, excluding the grouping columns
df = df.groupby(['mmsi', 'Voyage_ID'], group_keys=False).apply(calculate_differences).reset_index(drop=True)

C:\Users\Shamong\AppData\Local\Temp\ipykernel_9396\3714281646.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby(['mmsi', 'Voyage_ID'], group_keys=False).apply(calculate_differences).reset_index(drop=True)


In [30]:
# Save the updated dataframe to a new file
output_file_path = "D:\\AIS Project\\datasets\\voyage_time_distance_differences.csv"
df.to_csv(output_file_path, index=False)

print(f"Time and distance differences calculated and saved to {output_file_path}")

Time and distance differences calculated and saved to D:\AIS Project\datasets\voyage_time_distance_differences.csv
